# 17 — Grouped CV protocol v2 (YOLO26n/s-seg × control/offline_aug)

โน้ตบุ๊กแยกจาก `16_train_last.ipynb` เฉพาะ pipeline กลุ่ม v2: grouped development CV 5 folds, parent-uniform sampling, offline recipe `color_light/clahe_blur/environment` + online recipe (hflip/affine/mosaic/copy-paste), freeze gate และ evaluation แยกขั้น

`16_train_last.ipynb` เก็บไว้เป็น historical context (fixed-split benchmark 6 โมเดล) และเป็น launcher ของ run ชุด `3b867e9545011a5f3c52` — เล่มนี้ adopt protocol นั้นได้ผ่าน `NB16_ADOPT_PLAN=1` + `NB16_APPROVED_PROTOCOL=3b867e9545011a5f3c52` โดยตรวจ hash ของ plan/spec ทุกชั้น ไม่มีการยอมรับ artifact โดยไม่ตรวจ

## 1. Setup — paths, flags และ package versions

In [1]:
import os, json, time, hashlib, platform, random, itertools, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

EXP = next((p for p in [Path.cwd(), Path.cwd()/'experiment', Path.home()/'Desktop/Aggregate_Project/experiment'] if (p/'data/gold/split_audit.csv').is_file()), None)
if EXP is None:
    raise FileNotFoundError('Set EXP to the experiment directory containing data/gold/split_audit.csv.')
DATA = EXP/'data/gold_fixed_v1'
CLS = ['gravel','sand']
SEEDS = [42, 123, 2026]
DO_TRAIN = False
DO_INFER = False
DO_EVAL_TEST = False
DO_BOOTSTRAP = False
DO_EXPORT = False
IMGSZ = 960
CONF_FLOOR = 0.001
MAX_DETS = 100
EVAL_VERSION = 'nb16-v3-box-mask-numeric-cache'
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = EXP/'results/nb16_v2'/STAMP
OUT.mkdir(parents=True, exist_ok=False)
CACHE_ROOT = EXP/'results/nb16_v2/cache'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT = EXP/'runs/nb16_v2'

def file_sha(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1<<20), b''): h.update(block)
    return h.hexdigest()

def write_json(path, data):
    Path(path).write_text(json.dumps(data, indent=2, allow_nan=False, default=lambda x: x.item() if isinstance(x,np.generic) else str(x)), encoding='utf-8')

def savefig(name):
    plt.gcf().savefig(OUT/(name+'.pdf'), bbox_inches='tight')
    plt.gcf().savefig(OUT/(name+'.png'), dpi=300, bbox_inches='tight')
    plt.show(); plt.close()

plt.rcParams.update({'figure.dpi':90,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
versions = {}
for pkg in ['numpy','pandas','opencv-python','matplotlib','torch','torchvision','ultralytics','scipy','scikit-image','pycocotools','onnx','onnxruntime']:
    try: versions[pkg] = importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError: versions[pkg] = 'NOT INSTALLED'
write_json(OUT/'environment.json', {'python':platform.python_version(),'platform':platform.platform(),'packages':versions})
print('Protocol:', DATA, '| output:', OUT)
print('No training/inference/test evaluation enabled by default.')

Protocol: /home/supawich/Desktop/Aggregate_Project/experiment/data/gold_fixed_v1 | output: /home/supawich/Desktop/Aggregate_Project/experiment/results/nb16_v2/20261014T084814284700Z
No training/inference/test evaluation enabled by default.


## 2. Dataset manifest, nb00 handoff และ GT loader

In [2]:
def dataset_manifest(root):
    root = Path(root); rows=[]
    audit = pd.read_csv(root/'split_audit.csv')
    assert {'image','new_split','merged_group'}<=set(audit) and audit.image.is_unique
    assert audit[['image','new_split','merged_group']].notna().all().all()
    mapping = audit.set_index('image')
    for split in ('train','valid','test'):
        ips = sorted(p for p in (root/split/'images').iterdir() if p.suffix.lower() in {'.png','.jpg','.jpeg','.bmp'})
        expected = {p.stem for p in ips}; labels = {p.stem for p in (root/split/'labels').glob('*.txt')}
        assert ips and len(expected)==len(ips) and expected==labels, f'{split}: image/label mismatch or empty split'
        for ip in ips:
            assert ip.name in mapping.index and mapping.loc[ip.name,'new_split']==split, f'Split mapping differs: {ip.name}'
            im = cv2.imread(str(ip)); assert im is not None, f'Corrupt image: {ip}'
            lt = root/split/'labels'/(ip.stem+'.txt'); count=[0,0]
            for line in lt.read_text().splitlines():
                p=line.split(); assert len(p)>=7 and (len(p)-1)%2==0
                c=int(p[0]); xy=np.asarray(p[1:],float).reshape(-1,2)
                assert c in (0,1) and np.isfinite(xy).all() and ((xy>=0)&(xy<=1)).all()
                area=.5*abs(np.dot(xy[:,0],np.roll(xy[:,1],1))-np.dot(xy[:,1],np.roll(xy[:,0],1)))
                assert len(np.unique(xy,axis=0))>=3 and area>0
                count[c]+=1
            rows.append(dict(stem=ip.stem,image=ip.name,path=str(ip),split=split,group=str(mapping.loc[ip.name,'merged_group']),image_sha=file_sha(ip),label_sha=file_sha(lt),height=im.shape[0],width=im.shape[1],gravel=count[0],sand=count[1],decoded_sha=hashlib.sha256(im.tobytes()+str(im.shape).encode()).hexdigest()))
    frame=pd.DataFrame(rows); assert frame.stem.is_unique
    assert frame.groupby('group').split.nunique().max()==1, 'Merged-group leakage'
    assert frame.groupby('image_sha').split.nunique().max()==1, 'Exact duplicate crosses splits'
    assert frame.groupby('decoded_sha').split.nunique().max()==1, 'Same decoded pixels cross splits'
    identity=hashlib.sha256(frame.sort_values('stem')[['stem','split','group','image_sha','label_sha']].to_csv(index=False).encode()).hexdigest()
    return frame,identity

MANIFEST,DATA_ID=dataset_manifest(DATA)
GROUP_MAP=MANIFEST.set_index('stem').group.to_dict()
MANIFEST.to_csv(OUT/'dataset_manifest.csv',index=False)
print(MANIFEST.groupby('split').agg(images=('stem','size'),groups=('group','nunique'),gravel=('gravel','sum'),sand=('sand','sum')))
print('Dataset identity:',DATA_ID)
HANDOFF=None; QUALITY_PATH=None
handoffs=sorted((EXP/'results/nb00_audit').glob('*/handoff.json'),reverse=True)
nb00=json.loads((EXP/'00_EDA.ipynb').read_text())
nb00_source=hashlib.sha256(json.dumps([''.join(c['source']) for c in nb00['cells'] if c['cell_type']=='code'],ensure_ascii=False).encode()).hexdigest()
for path in handoffs:
    item=json.loads(path.read_text())
    if item.get('schema')!='nb00-to-nb16-v1' or item.get('dataset_id')!=DATA_ID or item.get('source_id')!=nb00_source: continue
    for field in ['manifest','quality','quality_reference']:
        artifact=path.parent/item[field]
        if artifact.resolve().parent!=path.parent.resolve() or file_sha(artifact)!=item[field+'_sha']: raise ValueError('Handoff artifact path/checksum mismatch')
    assert item['quality_threshold_source']=='train'
    HANDOFF=item; QUALITY_PATH=path.parent/item['quality']; break
if HANDOFF:
    write_json(OUT/'nb00_handoff_used.json',HANDOFF)
    print('Verified nb00 handoff:',QUALITY_PATH,'|',HANDOFF['geometry_status'],'| near-duplicate review:',HANDOFF['near_duplicate_review_status'])
else: print('GAP: run nb00 setup and authoritative final audit; training is blocked until a matching geometry-verified handoff exists.')

def load_gt(split_dir):
    result={}
    for ip in sorted((Path(split_dir)/'images').iterdir()):
        if ip.suffix.lower() not in {'.png','.jpg','.jpeg','.bmp'}: continue
        image=cv2.imread(str(ip)); assert image is not None
        h,w=image.shape[:2]; masks,classes=[],[]
        for line in (Path(split_dir)/'labels'/(ip.stem+'.txt')).read_text().splitlines():
            p=line.split(); xy=np.asarray(p[1:],float).reshape(-1,2)*[w,h]
            mask=np.zeros((h,w),np.uint8); cv2.fillPoly(mask,[xy.astype(np.int32)],1)
            assert mask.any(), f'Empty rasterized GT: {ip.name}'
            masks.append(mask.astype(bool)); classes.append(int(p[0]))
        result[ip.stem]=(ip,masks,classes)
    return result
GT_VALID=load_gt(DATA/'valid'); GT_TEST=load_gt(DATA/'test')
print('GT loaded; no model predictions evaluated.')

       images  groups  gravel  sand
split                              
test      105      12     189   159
train     753       7    1301  1550
valid     226       7     320   374
Dataset identity: 9bf2fab6eb34eb82dc7c4aad4b4de736c217e203c2f0f7cb2e36c6118705b8b9
Verified nb00 handoff: /home/supawich/Desktop/Aggregate_Project/experiment/results/nb00_audit/20261014T075628792047Z/final_quality_flags.csv | shapely validity checked | near-duplicate review: PENDING


GT loaded; no model predictions evaluated.


## 3. Canonical evaluator — COCO 101-point AP, threshold จาก validation เท่านั้น

In [3]:
def as_box(value):
    a = np.asarray(value)
    if a.ndim == 1:
        if a.shape != (4,) or not np.isfinite(a).all() or (a[2:] <= a[:2]).any():
            raise ValueError('Expected finite positive-area xyxy box')
        return a.astype(float)
    if a.ndim != 2 or not a.any():
        raise ValueError('Expected nonempty HxW mask')
    ys,xs = np.where(a.astype(bool))
    return np.array([xs.min(),ys.min(),xs.max()+1,ys.max()+1],float)

def iou_mat(pm,gm):
    out = np.zeros((len(pm),len(gm)),float)
    for i,p in enumerate(pm):
        for j,g in enumerate(gm):
            if np.asarray(p).ndim == 1 or np.asarray(g).ndim == 1:
                a,b = as_box(p),as_box(g)
                inter = np.maximum(0,np.minimum(a[2:],b[2:])-np.maximum(a[:2],b[:2])).prod()
                union = (a[2:]-a[:2]).prod()+(b[2:]-b[:2]).prod()-inter
            else:
                a,b = np.asarray(p,bool),np.asarray(g,bool)
                if a.shape != b.shape: raise ValueError('Prediction/GT shape mismatch')
                inter = (a&b).sum(); union = (a|b).sum()
            out[i,j] = inter/union if union else 0.
    return out

def validate_predictions(preds,gt,task=None):
    if set(preds) != set(gt): raise ValueError('Prediction IDs must exactly cover the evaluation cohort, including empty results')
    for st,pl in preds.items():
        ip,gm,gc = gt[st]
        if len(gm) != len(gc): raise ValueError('GT mask/class length mismatch')
        shape = gm[0].shape if gm else cv2.imread(str(ip)).shape[:2]
        for c,s,m in pl:
            if c not in (0,1) or int(c) != c or not np.isfinite(s) or not 0 <= s <= 1:
                raise ValueError('Invalid prediction class/confidence')
            a = np.asarray(m)
            if a.ndim == 1:
                box = as_box(a)
                if task == 'segment' or (box[:2]<0).any() or box[2]>shape[1] or box[3]>shape[0]:
                    raise ValueError('Wrong representation or out-of-image box')
            elif a.ndim != 2 or a.shape != shape or not np.isfinite(a).all() or not a.any() or not np.isin(a,[0,1]).all() or task == 'detect':
                raise ValueError('Wrong representation or invalid binary mask')

def match_image(pl,gm,gc,threshold=.5,conf=0.,class_aware=True,max_dets=MAX_DETS):
    ordered = sorted(enumerate(pl),key=lambda x:-float(x[1][1]))
    used,counts,records = set(),{},[]
    for pi,(c,s,m) in ordered:
        if c not in (0,1) or not np.isfinite(s) or not 0<=s<=1: raise ValueError('Invalid prediction class/score')
        if s < conf: continue
        counts[c] = counts.get(c,0)+1
        if counts[c] > max_dets: continue
        candidates = [j for j in range(len(gm)) if j not in used and (not class_aware or gc[j]==c)]
        overlaps = iou_mat([m],[gm[j] for j in candidates])[0] if candidates else np.array([])
        j = candidates[int(overlaps.argmax())] if len(overlaps) and overlaps.max() >= threshold else None
        if j is not None: used.add(j)
        records.append(dict(pred=pi,cls=int(c),score=float(s),gt=j,tp=int(j is not None),iou=float(overlaps.max()) if len(overlaps) else 0.))
    return records,used

def pr_curve(preds,gt,c,iou=.5):
    if set(preds)-set(gt): raise ValueError('Unknown prediction image IDs')
    n = sum(sum(x==c for x in gc) for _,gm,gc in gt.values())
    records = []
    for st,(_,gm,gc) in gt.items():
        matched,_ = match_image(preds.get(st,[]),gm,gc,iou)
        records.extend(r for r in matched if r['cls']==c)
    records.sort(key=lambda r:-r['score'])
    tp = np.cumsum([r['tp'] for r in records],dtype=float)
    precision = tp/np.arange(1,len(tp)+1) if len(tp) else np.array([])
    recall = tp/n if n else np.zeros(len(tp))
    envelope = np.maximum.accumulate(precision[::-1])[::-1] if len(tp) else precision
    samples = np.zeros(101)
    for i,r in enumerate(np.linspace(0,1,101)):
        ids = np.flatnonzero(recall >= r)
        if len(ids): samples[i] = envelope[ids[0]]
    return dict(gt=n,ap=float(samples.mean()) if n else None,precision=precision,recall=recall,scores=np.array([r['score'] for r in records]),samples=samples)

def eval_reference(preds,gt):
    out = {}
    for c,cn in enumerate(CLS):
        aps = [pr_curve(preds,gt,c,t)['ap'] for t in np.linspace(.5,.95,10)]
        out[cn] = {'AP':float(np.mean(aps)) if aps[0] is not None else None,'AP50':aps[0],'AP75':aps[5]}
    active = [v['AP'] for v in out.values() if v['AP'] is not None]
    out['AP_macro'] = float(np.mean(active)) if active else None
    return out

def operating_metrics(preds,gt,thresholds):
    rows = []
    for c,cn in enumerate(CLS):
        tp=fp=fn=0; size={b:[0,0] for b in ['small','medium','large']}
        for st,(_,gm,gc) in gt.items():
            pl = [p for p in preds.get(st,[]) if p[0]==c]
            rr,used = match_image(pl,gm,gc,.5,float(thresholds[c]))
            tp += len(used); fp += len(rr)-len(used)
            ids = [j for j,x in enumerate(gc) if x==c]; fn += len(ids)-len(used)
            for j in ids:
                a = np.asarray(gm[j],bool).sum(); b='small' if a<1024 else ('medium' if a<9216 else 'large')
                size[b][1] += 1; size[b][0] += int(j in used)
        rows.append(dict(cls=cn,threshold=float(thresholds[c]),tp=tp,fp=fp,fn=fn,precision=tp/(tp+fp) if tp+fp else None,recall=tp/(tp+fn) if tp+fn else None,F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else None,**{b+'_n':v[1] for b,v in size.items()},**{b+'_recall':v[0]/v[1] if v[1] else None for b,v in size.items()}))
    return pd.DataFrame(rows)

def select_thresholds(preds,valid):
    thresholds = {}
    for c in (0,1):
        curve = pr_curve(preds,valid,c)
        if not curve['gt']: raise ValueError('Validation must contain both classes for threshold selection')
        if not len(curve['scores']): thresholds[c]=float(np.nextafter(1.,2.)); continue
        scores = curve['scores']; last = np.r_[np.diff(scores)!=0,True]
        f = 2*curve['precision']*curve['recall']/np.maximum(curve['precision']+curve['recall'],1e-12)
        ids = np.flatnonzero(last); thresholds[c] = float(scores[ids[np.argmax(f[ids])]])
    return thresholds

def boundary_iou(p,g,ratio=.02):
    p,g=np.asarray(p,bool),np.asarray(g,bool)
    if p.shape!=g.shape: raise ValueError('Boundary mask shape mismatch')
    d=max(1,int(round(ratio*np.hypot(*g.shape))))
    def band(mask):
        padded=np.pad(mask.astype(np.uint8),d)
        eroded=cv2.erode(padded,np.ones((3,3),np.uint8),iterations=d)[d:-d,d:-d].astype(bool)
        return mask & ~eroded
    a,b=band(p),band(g); u=(a|b).sum()
    return float((a&b).sum()/u) if u else 0.

In [4]:
def canonical_coco(preds,gt,mode='segm'):
    if mode not in ('segm','bbox'): raise ValueError('Unsupported COCO mode')
    validate_predictions(preds,gt,'segment' if mode=='segm' else None)
    try:
        from pycocotools.coco import COCO
        from pycocotools.cocoeval import COCOeval
        from pycocotools import mask as mask_utils
    except ImportError:
        return {'status':'BLOCKED: pycocotools missing; reference metrics are diagnostic only'}
    import contextlib, io
    coco = COCO(); images=[]; anns=[]; detections=[]; aid=1
    for iid,(st,(ip,gm,gc)) in enumerate(gt.items(),1):
        shape=gm[0].shape if gm else cv2.imread(str(ip)).shape[:2]
        images.append(dict(id=iid,height=int(shape[0]),width=int(shape[1])))
        for m,c in zip(gm,gc):
            rle=mask_utils.encode(np.asfortranarray(np.asarray(m,np.uint8)))
            anns.append(dict(id=aid,image_id=iid,category_id=int(c)+1,segmentation=rle,area=float(mask_utils.area(rle)),bbox=mask_utils.toBbox(rle).tolist(),iscrowd=0)); aid+=1
        for c,s,m in preds[st]:
            item=dict(image_id=iid,category_id=int(c)+1,score=float(s))
            if mode=='segm': item['segmentation']=mask_utils.encode(np.asfortranarray(np.asarray(m,np.uint8)))
            else:
                box=as_box(m); item['bbox']=[float(box[0]),float(box[1]),float(box[2]-box[0]),float(box[3]-box[1])]
            detections.append(item)
    coco.dataset=dict(info={},images=images,annotations=anns,categories=[dict(id=i+1,name=c) for i,c in enumerate(CLS)])
    with contextlib.redirect_stdout(io.StringIO()):
        coco.createIndex()
        if detections: dt=coco.loadRes(detections)
        else:
            dt=COCO(); dt.dataset=dict(info={},images=images,annotations=[],categories=coco.dataset['categories']); dt.createIndex()
        ev=COCOeval(coco,dt,mode); ev.params.maxDets=[1,10,MAX_DETS]
        ev.evaluate(); ev.accumulate(); ev.summarize()
    names=['AP','AP50','AP75','AP_small','AP_medium','AP_large','AR1','AR10','AR100','AR_small','AR_medium','AR_large']
    result={k:float(v) if v>=0 else None for k,v in zip(names,ev.stats)}
    result['status']='canonical COCOeval'; result['mode']=mode; result['per_class']={}
    for i,cn in enumerate(CLS):
        p=ev.eval['precision'][:,:,i,0,-1]; values=p[p>=0]
        result['per_class'][cn]=float(values.mean()) if len(values) else None
    return result

im=Path('synthetic')
a=np.zeros((12,12),bool); a[:,:8]=1
b=np.zeros((12,12),bool); b[:,4:]=1
u=a|b
gt={'x':(im,[a,b],[0,0])}
assert len(match_image([(0,.9,a),(0,.8,u)],[a,b],[0,0])[1])==2
assert operating_metrics({'x':[(0,.9,u)]},gt,{0:0,1:0}).iloc[0].small_recall==.5
gt10={'x':(im,[a.copy() for _ in range(10)],[0]*10)}
assert np.isclose(pr_curve({'x':[(0,.9,a)]},gt10,0)['ap'],11/101)
assert pr_curve({},gt10,0)['ap']==0 and pr_curve({},gt10,1)['ap'] is None
assert np.isclose(boundary_iou(a,a),1)
assert match_image([(1,.9,a)],[a],[0])[1]==set()
assert match_image([(0,.9,a),(0,.8,a)],[a],[0])[1]=={0}
assert np.isclose(pr_curve({'x':[(0,.9,a),(0,.8,a)]},{'x':(im,[a],[0])},0)['ap'],1)
assert np.isclose(iou_mat([as_box(a)],[a])[0,0],1)
assert iou_mat([np.array([9,0,12,12])],[a])[0,0]==0
validate_predictions({'x':[(0,.9,as_box(a))]},gt10,'detect')
for bad in ({}, {'x':[(0,np.nan,a)]}, {'x':[(0,.5,np.zeros_like(a))]}):
    try: validate_predictions(bad,gt10)
    except ValueError: pass
    else: raise AssertionError('Invalid prediction contract accepted')
for mode,preds in [('segm',{'x':[(0,.9,a)]}),('bbox',{'x':[(0,.9,as_box(a))]})]:
    official=canonical_coco(preds,gt10,mode)
    if official['status']=='canonical COCOeval':
        assert np.isclose(official['AP'],eval_reference(preds,gt10)['AP_macro'],atol=1e-8)
    print(mode,official['status'])
print('Evaluator regression assertions passed; canonical parity is checked only when COCOeval is installed.')

segm canonical COCOeval
bbox canonical COCOeval
Evaluator regression assertions passed; canonical parity is checked only when COCOeval is installed.


## 4. Recipes (2 โมเดล), code identity และ protocol continuity

In [5]:
RECIPES = [
    dict(name='yolo26n_seg',kind='yolo',task='segment',init='yolo26n-seg.pt',size=960,epochs=100,batch=8,lr=.001),
    dict(name='yolo26s_seg',kind='yolo',task='segment',init='yolo26s-seg.pt',size=960,epochs=100,batch=8,lr=.001),
]
write_json(OUT/'recipes.json',RECIPES)

nb_source = json.loads((EXP/'17_cv_v2.ipynb').read_text())
CODE_ID = hashlib.sha256(json.dumps([''.join(c.get('source',[])) for c in nb_source['cells'] if c['cell_type']=='code'],ensure_ascii=False).encode()).hexdigest()

# Protocol launched from 16_train_last.ipynb before this notebook existed.
# Runs under runs/nb16_recipe_v2/3b867e9545011a5f3c52 carry that notebook's code identity;
# adopt it (with hash verification) when continuing that launch, never silently.
V2_LAUNCHED = dict(plan_id='3b867e9545011a5f3c52',source_id='c2bfcca6de9b7923f982158a5ed60accd070580bd6b629d5d2b83c6a05637d00',notebook='16_train_last.ipynb')
ACCEPTED_SOURCE_IDS = {CODE_ID,V2_LAUNCHED['source_id']}
ACCEPTED_PLAN_IDS = {V2_LAUNCHED['plan_id']}

def cv_adopt_plan(token):
    """Load a previously launched plan from results/nb16_recipe_v2/<token>/plan.json with full identity checks."""
    path=(EXP/'results/nb16_recipe_v2'/token/'plan.json')
    assert path.is_file(), f'No launched plan recorded for {token}'
    plan=json.loads(path.read_text())
    assert cv_json_id(plan)[:20]==token and plan['source_id'] in ACCEPTED_SOURCE_IDS
    assert plan['schema']=='nb16-parent-recipe-v2' and plan['cv_id']==CV_HANDOFF['cv_id'] and plan['dataset_id']==DATA_ID
    return plan

def runtime():
    import torch
    if not hasattr(torch,'cuda'):
        raise RuntimeError('This kernel has an incomplete torch installation. Select the training environment; no automatic environment changes.')
    return torch, 'cuda:0' if torch.cuda.is_available() else 'cpu'

def seed_everything(seed):
    torch,device=runtime()
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    if hasattr(torch.backends,'cudnn'):
        torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
    return torch,device

print(pd.DataFrame(RECIPES)[['name','init','size','epochs','batch','lr']].to_string(index=False))


       name           init  size  epochs  batch    lr
yolo26n_seg yolo26n-seg.pt   960     100      8 0.001
yolo26s_seg yolo26s-seg.pt   960     100      8 0.001


## 5. Predict/ensure/load model — YOLO segmentation เท่านั้น

In [6]:
def predict_model(model,recipe,gt,post=None):
    assert recipe['kind']=='yolo' and recipe['task']=='segment', 'v2 notebook supports YOLO segmentation recipes only'
    result={}
    torch,device=runtime()
    for st,(ip,gm,gc) in gt.items():
        im=cv2.imread(str(ip)); h,w=im.shape[:2]
        r=model.predict(im,imgsz=recipe['size'],conf=CONF_FLOOR,max_det=MAX_DETS,retina_masks=True,device=device,verbose=False)[0]
        result[st]=[] if r.masks is None else [(int(c),float(s),cv2.resize((m>.5).astype(np.uint8),(w,h),interpolation=cv2.INTER_NEAREST).astype(bool)) for c,s,m in zip(r.boxes.cls.cpu().numpy(),r.boxes.conf.cpu().numpy(),r.masks.data.cpu().numpy())]
    return result

def ensure_coco():
    import importlib.util
    if importlib.util.find_spec('pycocotools') is None:
        raise RuntimeError('Canonical model selection requires pycocotools in this kernel.')

def load_model(recipe,path):
    assert recipe['kind']=='yolo', 'v2 notebook supports YOLO recipes only'
    from ultralytics import YOLO
    return YOLO(str(path))


## 6. CV handoff gate — รับเฉพาะ `nb00-groupcv-v2`

In [7]:
CV_ARMS = ['control','offline_aug']
DO_CV_SMOKE = os.environ.get('NB16_CV_SMOKE','0') == '1'
CV_HANDOFF = None
CV_ROOT = None
CV_LINEAGE = None
CV_PREFLIGHT = []
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')


def check_cv_group_roles(frame):
    assert frame.groupby('cv_group').role.nunique().eq(1).all(), 'CV group crosses roles'


def load_cv_handoff():
    source=MANIFEST.set_index('stem')
    for path in sorted((EXP/'results/nb00_audit').glob('*/cv_handoff.json'),reverse=True):
        item=json.loads(path.read_text())
        if item.get('schema')!='nb00-groupcv-v2' or item.get('source_dataset_id')!=DATA_ID or item.get('source_code_id')!=nb00_source: continue
        assert item['status']=='PASS' and item['geometry_status']=='shapely validity checked'
        root=Path(item['cv_root']).resolve()
        assert root.parent==(EXP/'data').resolve() and root.name.startswith('groupcv_offline_v2_')
        manifest_path=root/'manifest.json'
        assert file_sha(manifest_path)==item['manifest_sha']
        manifest=json.loads(manifest_path.read_text())
        assert manifest['spec']==item['protocol'] and manifest['cv_id']==item['cv_id']
        assert hashlib.sha256(json.dumps(manifest['spec'],sort_keys=True).encode()).hexdigest()==item['cv_id']
        assert manifest['spec']['schema']=='groupcv-offline-v2' and item['folds']==manifest['spec']['k']==5
        assert manifest['spec']['augmentation']['variants']==['color_light','clahe_blur','environment']
        assert manifest['spec']['sampling']==dict(parent='uniform_with_replacement',aug_variant_probabilities={v:.25 for v in ['original','color_light','clahe_blur','environment']},control_variant='original',hard_scene_oversampling=False)
        for name,sha in manifest['artifacts'].items():
            artifact=(root/name).resolve()
            assert artifact.is_relative_to(root) and file_sha(artifact)==sha
        assignments=pd.read_csv(root/'assignments.csv',keep_default_na=False)
        assert set(assignments.fold)==set(range(item['folds']))
        assert hashlib.sha256(assignments.sort_values(['fold','stem']).to_csv(index=False).encode()).hexdigest()==manifest['spec']['assignments_sha']
        held=assignments[assignments.role=='heldout']
        assert set(held.stem)==set(source.index) and held.stem.value_counts().eq(1).all()
        lineage=pd.read_csv(root/'lineage.csv',keep_default_na=False)
        assert set(lineage.arm)==set(CV_ARMS) and set(lineage.fold)==set(range(item['folds'])) and not lineage.duplicated(['fold','arm','image']).any()
        reports=[]
        for fold in range(item['folds']):
            frame=assignments[assignments.fold==fold]
            assert frame.stem.is_unique and set(frame.stem)==set(source.index) and set(frame.role)=={'train','valid','heldout'}
            check_cv_group_roles(frame)
            assert frame.groupby('original_group').role.nunique().eq(1).all()
            assert frame.groupby('role')[['gravel','sand']].sum().gt(0).all().all()
            for field in ['upload_uuid','image_sha','decoded_sha']:
                subset=frame[frame[field]!='']; assert subset.groupby(field).role.nunique().eq(1).all()
            for field in ['image_sha','label_sha','decoded_sha','gravel','sand']:
                assert frame.set_index('stem')[field].sort_index().equals(source[field].sort_index())
            assert frame.set_index('stem').original_group.to_dict()==source.group.to_dict()
            reference=json.loads((root/f'quality_reference_fold{fold}.json').read_text())
            assert reference['selected_on']=='inner_train' and set(reference['source_stems'])==set(frame.loc[frame.role=='train','stem'])
            assignment=frame.set_index('stem')
            for arm in CV_ARMS:
                sub=lineage[(lineage.fold==fold)&(lineage.arm==arm)]
                assert sub.groupby('cv_group').role.nunique().eq(1).all()
                for role,split in [('train','train'),('valid','valid'),('heldout','test')]:
                    rows=sub[sub.role==role]; expected=set(frame.loc[frame.role==role,'stem'])
                    variants={'original','color_light','clahe_blur','environment'} if arm=='offline_aug' and role=='train' else {'original'}
                    assert set(rows.parent_stem)==expected and len(rows)==len(expected)*len(variants)
                    assert rows.groupby('parent_stem').variant.apply(set).map(lambda x:x==variants).all()
                    directory=root/f'fold{fold}'/arm/split
                    assert set(p.name for p in (directory/'images').iterdir())==set(rows.image)
                    assert set(p.stem for p in (directory/'labels').glob('*.txt'))==set(rows.stem)
                    for r in rows.itertuples(index=False):
                        original=source.loc[r.parent_stem]; owner=assignment.loc[r.parent_stem]
                        assert r.role==owner.role and r.cv_group==owner.cv_group and r.original_group==owner.original_group
                        assert r.source_image_sha==original.image_sha and r.source_label_sha==original.label_sha
                        for name,sha,kind in [(r.image_path,r.image_sha,'images'),(r.label_path,r.label_sha,'labels')]:
                            artifact=(root/name).resolve()
                            assert artifact.parent==(directory/kind).resolve() and file_sha(artifact)==sha
                        assert r.label_sha==original.label_sha
                        if r.variant=='original': assert r.image_sha==original.image_sha
                    reports.append(dict(fold=fold,arm=arm,role=role,images=len(rows),parents=len(expected),status='PASS'))
        return item,root,lineage,reports
    return None,None,None,[]


CV_HANDOFF,CV_ROOT,CV_LINEAGE,CV_PREFLIGHT=load_cv_handoff()
if CV_HANDOFF is None:
    print('PENDING v2: run 00 audit/materialization; v1 handoff is intentionally not accepted.')
    if DO_CV_SMOKE: raise RuntimeError('CV smoke blocked: missing/stale v2 handoff')
else:
    write_json(OUT/'cv_handoff_used.json',CV_HANDOFF)
    pd.DataFrame(CV_PREFLIGHT).to_csv(OUT/'cv_preflight.csv',index=False)
    print('PASS audited v2 handoff:',CV_ROOT); display(pd.DataFrame(CV_PREFLIGHT))
try:
    check_cv_group_roles(pd.DataFrame({'cv_group':['same','same'],'role':['train','heldout']}))
except AssertionError: print('PASS regression: cross-role group rejected')
else: raise AssertionError('Leakage regression did not reject invalid cohort')

PASS audited v2 handoff: /home/supawich/Desktop/Aggregate_Project/experiment/data/groupcv_offline_v2_de6276f231115417


,fold,arm,role,images,parents,status
0,0,control,train,737,737,PASS
1,0,control,valid,183,183,PASS
2,0,control,heldout,164,164,PASS
3,0,offline_aug,train,2948,737,PASS
4,0,offline_aug,valid,183,183,PASS
5,0,offline_aug,heldout,164,164,PASS
6,1,control,train,543,543,PASS
7,1,control,valid,140,140,PASS
8,1,control,heldout,401,401,PASS
9,1,offline_aug,train,2172,543,PASS


PASS regression: cross-role group rejected


### 6.1 Helper สำหรับเลือก validation parents ใน smoke

In [8]:
CV_SMOKE_RESULTS = []
SMOKE_MAX_STEPS = 2


def smoke_parents(frame, role, limit=4):
    rows = frame[(frame.role==role)&(frame.variant=='original')].sort_values('parent_stem')
    selected = []
    for cls in CLS:
        for stem in rows.parent_stem:
            if MANIFEST.set_index('stem').loc[stem,cls] > 0 and stem not in selected:
                selected.append(stem)
                if sum(MANIFEST.set_index('stem').loc[s,cls] > 0 for s in selected) >= 2: break
    for stem in rows.parent_stem:
        if len(selected) >= limit: break
        if stem not in selected: selected.append(stem)
    assert len(selected) <= limit and MANIFEST.set_index('stem').loc[selected,CLS].sum().gt(0).all()
    return selected


print('Legacy batch-2 smoke helper retained. Production-batch v2 smoke executes in cell 43 when NB16_CV_SMOKE=1.')

Legacy batch-2 smoke helper retained. Production-batch v2 smoke executes in cell 43 when NB16_CV_SMOKE=1.


## 7. Parent-uniform sampler, online recipe และ production loader

In [9]:
import math, gc, copy, gzip
from collections import Counter

DO_FULL_CV = os.environ.get('NB16_FULL_CV','0') == '1'
DO_CV_PREVIEW = os.environ.get('NB16_CV_PREVIEW','0') == '1'
DO_CV_FREEZE = os.environ.get('NB16_CV_FREEZE','0') == '1'
DO_CV_EVALUATE = os.environ.get('NB16_CV_EVALUATE','0') == '1'
CV_APPROVED_PROTOCOL = os.environ.get('NB16_APPROVED_PROTOCOL','')
CV_APPROVED_PREVIEW_SHA = os.environ.get('NB16_APPROVED_PREVIEW_SHA','')
CV_FULL_SEEDS = [42]
CV_FULL_EPOCHS = 30
CV_BATCH_SIZES = {'yolo26n_seg':12,'yolo26s_seg':6}
CV_FULL_RECIPES = [dict(r,epochs=CV_FULL_EPOCHS,batch=CV_BATCH_SIZES[r['name']]) for r in RECIPES if r['name'] in CV_BATCH_SIZES]
CV_VARIANTS = ['original','color_light','clahe_blur','environment']
CV_ONLINE = dict(fliplr=.5,flipud=0.,degrees=10.,translate=.05,scale=.1,shear=3.,perspective=0.,mosaic=.25,copy_paste=.10,copy_paste_mode='mixup',copy_paste_fraction=1.,close_mosaic=0,mixup=0.,cutmix=0.,hsv_h=0.,hsv_s=0.,hsv_v=0.,bgr=0.,crop='affine clipping only',albumentations=False)
CV_TRAIN_ARGS = dict(optimizer='AdamW',lr0=.001,lrf=.01,cos_lr=False,momentum=.937,weight_decay=.0005,warmup_epochs=0,amp=False,deterministic=True,workers=0,cache=False,plots=False,save=True,overlap_mask=True,mask_ratio=4,channels_last=False,compile=False)


def cv_json_id(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,allow_nan=False).encode()).hexdigest()


def cv_atomic_json(path,value):
    path=Path(path); temporary=path.with_name(path.name+'.tmp')
    write_json(temporary,value); os.replace(temporary,path)


def cv_atomic_torch(path,value):
    torch,_=runtime(); path=Path(path); temporary=path.with_name(path.name+'.tmp')
    torch.save(value,temporary); os.replace(temporary,path)


class CVParentSampler:
    def __init__(self,rows,seed,augmented):
        assert not rows.empty and rows.role.eq('train').all() and not rows.duplicated(['parent_stem','variant']).any()
        self.parents=sorted(rows.parent_stem.unique())
        self.variants=CV_VARIANTS if augmented else ['original']
        assert rows.groupby('parent_stem').variant.apply(set).map(lambda x:x==set(self.variants)).all()
        self.files={(r.parent_stem,r.variant):r.image for r in rows.itertuples(index=False)}
        self.parent_rng=np.random.default_rng(seed)
        self.donor_rng=np.random.default_rng(seed+1)
        self.variant_rng=np.random.default_rng(seed+2)
        self.counts=Counter(); self.primary_counts=Counter(); self.outputs=0; self.draws=[]
    def draw(self,primary=False):
        rng=self.parent_rng if primary else self.donor_rng
        parent=self.parents[int(rng.integers(len(self.parents)))]
        variant=self.variants[int(self.variant_rng.integers(len(self.variants)))] if len(self.variants)>1 else 'original'
        key=(parent,variant); self.counts[key]+=1
        if primary: self.primary_counts[key]+=1
        self.draws.append(dict(parent=parent,variant=variant,primary=primary))
        return self.files[key]
    def state_dict(self):
        return dict(parent_rng=copy.deepcopy(self.parent_rng.bit_generator.state),donor_rng=copy.deepcopy(self.donor_rng.bit_generator.state),variant_rng=copy.deepcopy(self.variant_rng.bit_generator.state),counts=dict(self.counts),primary_counts=dict(self.primary_counts),outputs=self.outputs)
    def load_state_dict(self,state):
        for name in ['parent_rng','donor_rng','variant_rng']:
            getattr(self,name).bit_generator.state=copy.deepcopy(state[name])
        self.counts=Counter(state['counts']); self.primary_counts=Counter(state['primary_counts']); self.outputs=state['outputs']; self.draws=[]
    def summary(self):
        return dict(output_images=self.outputs,source_parent_draws=sum(self.counts.values()),primary_parent_draws=sum(self.primary_counts.values()),donor_parent_draws=sum(self.counts.values())-sum(self.primary_counts.values()),unique_source_files=len(self.counts),unique_parents=len({p for p,v in self.counts}),variant_draws={v:sum(n for (p,k),n in self.counts.items() if k==v) for v in self.variants})
    def table(self):
        return pd.DataFrame([dict(parent_stem=p,variant=v,source_draws=self.counts[(p,v)],primary_draws=self.primary_counts[(p,v)],donor_draws=self.counts[(p,v)]-self.primary_counts[(p,v)]) for p in self.parents for v in self.variants])


def cv_parent_dataset(args,data,root,rows,seed,batch,steps,augmented):
    from ultralytics.data.dataset import YOLODataset
    from ultralytics.data.augment import Compose,LetterBox,Mosaic,RandomPerspective,RandomFlip,CopyPaste
    sampler=CVParentSampler(rows,seed,augmented)
    config=dict(CV_ONLINE) if augmented else dict(CV_ONLINE,fliplr=0.,degrees=0.,translate=0.,scale=0.,shear=0.,mosaic=0.,copy_paste=0.)

    class ParentDataset(YOLODataset):
        def __len__(self): return steps*batch if hasattr(self,'parent_sampler') else super().__len__()
        def get_image_and_label(self,index,primary=False):
            name=self.parent_sampler.draw(primary)
            return super().get_image_and_label(self.file_indexes[name])
        def __getitem__(self,index):
            self.parent_sampler.draws=[]
            result=self.transforms(self.get_image_and_label(index,primary=True))
            assert result['img'].shape==(3,args.imgsz,args.imgsz)
            assert len(result['cls'])==len(result['bboxes']) and result['masks'].ndim==3
            assert np.isfinite(result['bboxes'].numpy()).all() and np.isfinite(result['masks'].numpy()).all()
            self.parent_sampler.outputs+=1
            return result

    dataset=ParentDataset(img_path=str(Path(root)/'train/images'),imgsz=args.imgsz,batch_size=batch,augment=True,hyp=copy.copy(args),rect=False,cache=False,single_cls=False,stride=32,pad=0.,task='segment',data=data,fraction=1.)
    dataset.file_indexes={Path(name).name:i for i,name in enumerate(dataset.im_files)}
    assert set(dataset.file_indexes)==set(rows.image), 'Loader dropped/replaced files; audit labels first'
    lookup=rows.set_index('image')
    for label in dataset.labels:
        assert len(label['cls'])==int(lookup.loc[Path(label['im_file']).name,'instances']), 'Loader changed annotation count'
    dataset.parent_sampler=sampler
    formatter=dataset.transforms.transforms[-1]

    class ParentMosaic(Mosaic):
        def get_indexes(self): return [0,0,0]

    class DonorCopyPaste(CopyPaste):
        def get_indexes(self): return 0
        def __call__(self,labels):
            if random.random()>=config['copy_paste']: return labels
            return super().__call__(labels)

    affine=lambda: RandomPerspective(degrees=config['degrees'],translate=config['translate'],scale=config['scale'],shear=config['shear'],perspective=0.,size=(args.imgsz,args.imgsz))
    if augmented:
        dataset.transforms=Compose([ParentMosaic(dataset,imgsz=args.imgsz,p=config['mosaic']),affine(),DonorCopyPaste(dataset,pre_transform=Compose([affine()]),p=config['copy_paste_fraction'],mode='mixup'),RandomFlip(direction='horizontal',p=config['fliplr']),formatter])
    else: dataset.transforms=Compose([LetterBox(new_shape=(args.imgsz,args.imgsz),scaleup=True),formatter])
    dataset.pipeline_config=config
    return dataset


def cv_training_rows(fold,arm):
    rows=CV_LINEAGE[(CV_LINEAGE.fold==fold)&(CV_LINEAGE.arm==arm)&(CV_LINEAGE.role=='train')].copy()
    source=MANIFEST.set_index('stem')
    rows['instances']=rows.parent_stem.map((source.gravel+source.sand).to_dict())
    assert rows.instances.notna().all()
    return rows


def cv_plan():
    assert CV_HANDOFF is not None and len(CV_PREFLIGHT)==30, 'Build/audit v2 in 00 first'
    assert CV_HANDOFF['schema']=='nb00-groupcv-v2'
    assert CV_HANDOFF['protocol']['augmentation']['variants']==CV_VARIANTS[1:]
    assert CV_HANDOFF['protocol']['sampling']==dict(parent='uniform_with_replacement',aug_variant_probabilities={v:.25 for v in CV_VARIANTS},control_variant='original',hard_scene_oversampling=False)
    packages={p:importlib.metadata.version(p) for p in ['ultralytics','torch','torchvision','numpy','opencv-python','pycocotools']}
    assert packages['ultralytics']=='8.4.171', 'Review loader/resume adapter before changing the verified Ultralytics API'
    jobs=[dict(recipe=r,fold=f,seed=s,arm=a) for f in range(5) for r in CV_FULL_RECIPES for s in CV_FULL_SEEDS for a in CV_ARMS]
    plan=dict(schema='nb16-parent-recipe-v2',cv_id=CV_HANDOFF['cv_id'],dataset_id=DATA_ID,source_id=CODE_ID,packages=packages,jobs=jobs,offline=CV_HANDOFF['protocol']['augmentation'],online=CV_ONLINE,training=CV_TRAIN_ARGS,sampling=CV_HANDOFF['protocol']['sampling'],selection='inner-valid canonical mask AP; thresholds inner-valid; freeze before evaluation',scope='development grouped CV; one seed; recipe-level effect, not transform causality',resume='last completed virtual epoch; raw FP32 weights, optimizer, scheduler, scaler, EMA, criterion, RNG and parent sampler')
    return plan,cv_json_id(plan)[:20]

## 8. Training state, freeze และ evaluation แยกขั้นตอน

In [10]:
def cv_preview(plan,token):
    from ultralytics.cfg import get_cfg
    samples=[]; fig,axes=plt.subplots(2,4,figsize=(16,8))
    for row,arm in enumerate(CV_ARMS):
        seed_everything(42)
        args=get_cfg(overrides=dict(imgsz=960,task='segment',mask_ratio=CV_TRAIN_ARGS['mask_ratio'],overlap_mask=CV_TRAIN_ARGS['overlap_mask']))
        ds=cv_parent_dataset(args,{'names':{0:'gravel',1:'sand'}},CV_ROOT/'fold0'/arm,cv_training_rows(0,arm),42,2,8,arm=='offline_aug')
        for index in range(16):
            sample=ds[index]
            samples.append(dict(arm=arm,index=index,sources=copy.deepcopy(ds.parent_sampler.draws),image_sha=hashlib.sha256(sample['img'].numpy().tobytes()).hexdigest(),mask_sha=hashlib.sha256(sample['masks'].numpy().tobytes()).hexdigest()))
            if index<4:
                image=sample['img'].permute(1,2,0).numpy(); masks=sample['masks'].numpy()
                foreground=cv2.resize((masks.max(axis=0)>0).astype(np.uint8),(960,960),interpolation=cv2.INTER_NEAREST) if len(masks) else np.zeros((960,960),np.uint8)
                axes[row,index].imshow(image)
                if foreground.any(): axes[row,index].contour(foreground,levels=[.5],colors=['cyan'],linewidths=.6)
                axes[row,index].set_title(f'{arm}: {len(ds.parent_sampler.draws)} sources'); axes[row,index].axis('off')
    path=OUT/('cv_preview_'+token+'.json')
    write_json(path,dict(protocol_id=token,plan=plan,samples=samples,alignment_review='PENDING human review',training_executed=False))
    plt.tight_layout(); savefig('cv_production_preview_'+token)
    print('Review offline/online masks and realism. Approval IDs:',token,file_sha(path))
    return path


def cv_resume_load(path,spec):
    torch,_=runtime(); meta=Path(path).with_suffix('.json')
    assert meta.is_file(), 'Missing resume identity/checksum'
    stored=json.loads(meta.read_text())
    assert stored['spec']==spec and stored['checkpoint_sha']==file_sha(path), 'Resume identity/checksum differs'
    checkpoint=torch.load(path,map_location='cpu',weights_only=False)
    assert checkpoint['protocol_spec']==spec and checkpoint['epoch']>=0
    assert all(k in checkpoint for k in ['model','ema','optimizer','scheduler','scaler','rng','sampler','cv_steps','selection','timing','criterion_state','training_init_sha','selected_sha'])
    assert checkpoint['cv_steps']==(checkpoint['epoch']+1)*spec['steps_per_epoch']
    assert checkpoint['sampler']['outputs']==checkpoint['cv_steps']*spec['job']['recipe']['batch']
    assert checkpoint['selected_sha']==file_sha(Path(path).parent/'selected.pt'), 'Selected checkpoint differs from committed training state'
    return checkpoint


def cv_parameter_sha(model):
    digest=hashlib.sha256()
    for name,p in model.named_parameters(): digest.update(name.encode()); digest.update(p.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def cv_train_yolo(recipe,seed,run_dir,root,valid,steps_per_epoch,spec,report=None):
    from ultralytics import YOLO
    from ultralytics.models.yolo.segment.train import SegmentationTrainer
    torch,device=seed_everything(seed)
    resume_path=run_dir/'resume.pt'
    checkpoint=cv_resume_load(resume_path,spec) if resume_path.exists() else None
    if checkpoint is not None and checkpoint['epoch']+1==recipe['epochs']:
        return run_dir/'selected.pt',checkpoint['timing'],checkpoint['sampler'],checkpoint['training_init_sha']
    assert checkpoint is not None or not (run_dir/'selected.pt').exists(), 'Partial state missing; choose a new run identity instead of restarting silently'
    rows=cv_training_rows(spec['job']['fold'],spec['job']['arm']); augmented=spec['job']['arm']=='offline_aug'

    class MatchedTrainer(SegmentationTrainer):
        def get_dataloader(self,dataset_path,batch_size=16,rank=0,mode='train'):
            if mode!='train': return super().get_dataloader(dataset_path,batch_size,rank,mode)
            assert rank in (-1,0) and self.world_size==1 and self.args.workers==0
            ds=cv_parent_dataset(self.args,self.data,root,rows,seed,batch_size,steps_per_epoch,augmented)
            return torch.utils.data.DataLoader(ds,batch_size=batch_size,shuffle=False,num_workers=0,collate_fn=ds.collate_fn,drop_last=True)
        def _setup_train(self):
            super()._setup_train()
            self.last=resume_path; self.best=run_dir/'selected.pt'
            self.cv_steps=0; self.cv_selection=[]
            self.cv_timing=dict(training_seconds=0.,gpu_synchronized_batch_seconds=0.,gpu_elapsed_seconds=0.)
            self.training_init_sha=cv_parameter_sha(self.model)
            if checkpoint is not None:
                self.model.load_state_dict(checkpoint['model'].float().state_dict())
                self.scheduler.load_state_dict(checkpoint['scheduler'])
                self.model.criterion=self.model.init_criterion()
                for key,value in checkpoint['criterion_state'].items():
                    assert hasattr(self.model.criterion,key); setattr(self.model.criterion,key,value)
                self.cv_steps=checkpoint['cv_steps']; self.cv_selection=checkpoint['selection']; self.cv_timing=checkpoint['timing']; self.training_init_sha=checkpoint['training_init_sha']
                self.train_loader.dataset.parent_sampler.load_state_dict(checkpoint['sampler'])
                random.setstate(checkpoint['rng']['python']); np.random.set_state(checkpoint['rng']['numpy'])
                torch.set_rng_state(checkpoint['rng']['torch']); torch.cuda.set_rng_state_all(checkpoint['rng']['cuda'])
            assert self.batch_size==recipe['batch'] and self.accumulate==1 and not self.amp
        def optimizer_step(self):
            assert self.batch_size==recipe['batch'] and self.accumulate==1, 'Runtime changed paired budget'
            assert torch.isfinite(self.loss).all().item(), 'Non-finite loss'
            gradients=[p.grad for p in self.model.parameters() if p.grad is not None]
            assert gradients and any(torch.count_nonzero(g).item() for g in gradients), 'No nonzero gradients'
            torch.nn.utils.clip_grad_norm_(self.model.parameters(),10.,error_if_nonfinite=True)
            super().optimizer_step(); self.cv_steps+=1
        def validate(self):
            proxy=YOLO(str(EXP/'weights'/recipe['init'])); proxy.model=copy.deepcopy(self.ema.ema).float().eval()
            predictions=predict_model(proxy,recipe,valid); validate_predictions(predictions,valid,'segment')
            score=canonical_coco(predictions,valid,'segm')['AP']
            assert score is not None and np.isfinite(score) and self.cv_steps==(self.epoch+1)*steps_per_epoch
            if self.best_fitness is None or score>self.best_fitness:
                cv_atomic_torch(self.best,dict(model=copy.deepcopy(proxy.model).cpu().float(),train_args=vars(self.args),epoch=self.epoch,version=importlib.metadata.version('ultralytics')))
                self.best_fitness=score
            record=dict(epoch=self.epoch+1,optimizer_steps=self.cv_steps,val_AP=score,loss=float(self.loss.detach()))
            self.cv_selection.append(record); pd.DataFrame(self.cv_selection).to_csv(run_dir/'selection.csv',index=False)
            if report: report(record)
            del predictions,proxy; gc.collect(); torch.cuda.empty_cache()
            return dict(cv_val_AP=score),score
        def save_model(self):
            assert self.cv_steps==(self.epoch+1)*steps_per_epoch
            payload=dict(epoch=self.epoch,best_fitness=self.best_fitness,model=copy.deepcopy(self.model).cpu().float(),ema=copy.deepcopy(self.ema.ema).cpu().float(),updates=self.ema.updates,optimizer=self.optimizer.state_dict(),scheduler=self.scheduler.state_dict(),scaler=self.scaler.state_dict(),train_args=vars(self.args),version=importlib.metadata.version('ultralytics'),protocol_spec=spec,cv_steps=self.cv_steps,selection=self.cv_selection,timing=self.cv_timing,training_init_sha=self.training_init_sha,training_final_sha=cv_parameter_sha(self.model),selected_sha=file_sha(self.best),criterion_state={k:getattr(self.model.criterion,k) for k in ['updates','o2m','o2o','o2m_copy','final_o2m','total'] if hasattr(self.model.criterion,k)},sampler=self.train_loader.dataset.parent_sampler.state_dict(),rng=dict(python=random.getstate(),numpy=np.random.get_state(),torch=torch.get_rng_state(),cuda=torch.cuda.get_rng_state_all()))
            payload['model'].criterion=None; payload['ema'].criterion=None
            cv_atomic_torch(self.last,payload)
            cv_atomic_json(self.last.with_suffix('.json'),dict(spec=spec,checkpoint_sha=file_sha(self.last)))
            self.train_loader.dataset.parent_sampler.table().to_csv(run_dir/'sampling_counts.csv',index=False)
            write_json(run_dir/'sampling_summary.json',self.train_loader.dataset.parent_sampler.summary())
            return True
        def final_eval(self): return None

    cfg=run_dir/'data.yaml'
    cfg.write_text(f'path: {Path(root).resolve()}\ntrain: train/images\nval: valid/images\nnames: {CLS}\n')
    model=YOLO(str(resume_path if checkpoint is not None else EXP/'weights'/recipe['init']))
    def epoch_start(trainer):
        torch.cuda.synchronize(); trainer.cv_epoch_start=time.perf_counter()
    def batch_start(trainer):
        torch.cuda.synchronize(); trainer.cv_batch_start=time.perf_counter()
        trainer.cv_gpu_start=torch.cuda.Event(enable_timing=True); trainer.cv_gpu_end=torch.cuda.Event(enable_timing=True)
        trainer.cv_gpu_start.record()
    def batch_end(trainer):
        trainer.cv_gpu_end.record(); torch.cuda.synchronize()
        trainer.cv_timing['gpu_elapsed_seconds']+=trainer.cv_gpu_start.elapsed_time(trainer.cv_gpu_end)/1000.
        trainer.cv_timing['gpu_synchronized_batch_seconds']+=time.perf_counter()-trainer.cv_batch_start
    def epoch_end(trainer):
        torch.cuda.synchronize(); trainer.cv_timing['training_seconds']+=time.perf_counter()-trainer.cv_epoch_start
    for event,callback in [('on_train_epoch_start',epoch_start),('on_train_batch_start',batch_start),('on_train_batch_end',batch_end),('on_train_epoch_end',epoch_end)]: model.add_callback(event,callback)
    online={k:v for k,v in CV_ONLINE.items() if k not in ['copy_paste_fraction','crop','albumentations']}
    if not augmented: online.update(fliplr=0.,degrees=0.,translate=0.,scale=0.,shear=0.,mosaic=0.,copy_paste=0.)
    model.train(trainer=MatchedTrainer,data=str(cfg),epochs=recipe['epochs'],patience=recipe['epochs']+1,batch=recipe['batch'],nbs=recipe['batch'],imgsz=recipe['size'],device=device,seed=seed,split='val',auto_augment=None,erasing=0,project=str(run_dir),name='native',exist_ok=True,resume=str(resume_path) if checkpoint is not None else False,**CV_TRAIN_ARGS,**online)
    trainer=model.trainer
    assert trainer.cv_steps==spec['optimizer_steps'] and trainer.train_loader.dataset.parent_sampler.outputs==trainer.cv_steps*recipe['batch']
    assert cv_parameter_sha(trainer.model)!=trainer.training_init_sha, 'No model parameter update'
    result=(run_dir/'selected.pt',dict(trainer.cv_timing),trainer.train_loader.dataset.parent_sampler.state_dict(),trainer.training_init_sha)
    del model,trainer,checkpoint; gc.collect(); torch.cuda.empty_cache()
    return result


def cv_frozen_identity(spec,checkpoint,valid):
    return dict(spec=spec,checkpoint_sha=file_sha(checkpoint),threshold_source='inner-valid',valid_stems=sorted(valid),evaluator=EVAL_VERSION)


def cv_require_frozen(path,spec,checkpoint,valid):
    assert Path(path).is_file(), 'Run freeze stage before evaluation'
    frozen=json.loads(Path(path).read_text())
    assert frozen['identity']==cv_frozen_identity(spec,checkpoint,valid), 'Frozen model/config/validation differs'
    assert set(map(int,frozen['thresholds']))=={0,1} and all(np.isfinite(v) and 0<=v<=1 for v in frozen['thresholds'].values())
    return frozen


def cv_freeze(run_dir,spec,recipe,valid):
    path=run_dir/'freeze.json'; cp=run_dir/'selected.pt'; trained=json.loads((run_dir/'training.json').read_text())
    assert trained['spec']==spec and trained['checkpoint_sha']==file_sha(cp)
    if path.is_file(): return cv_require_frozen(path,spec,cp,valid)
    model=load_model(recipe,cp); predictions=predict_model(model,recipe,valid); validate_predictions(predictions,valid,'segment')
    frozen=dict(identity=cv_frozen_identity(spec,cp,valid),thresholds=select_thresholds(predictions,valid),created_utc=datetime.now(timezone.utc).isoformat(),status='frozen before heldout prediction')
    cv_atomic_json(path,frozen)
    del predictions,model; gc.collect(); runtime()[0].cuda.empty_cache()
    return frozen


def cv_save_predictions(predictions,path,task):
    from pycocotools import mask as mask_utils
    with gzip.open(path,'wt',encoding='utf-8') as stream:
        for stem,items in predictions.items():
            packed=[]
            for cls,score,value in items:
                if task=='segment':
                    value=mask_utils.encode(np.asfortranarray(np.asarray(value,np.uint8)))
                    value=dict(size=value['size'],counts=value['counts'].decode('ascii'))
                else: value=np.asarray(value,float).tolist()
                packed.append([int(cls),float(score),value])
            stream.write(json.dumps(dict(stem=stem,predictions=packed),allow_nan=False)+'\n')


def cv_evaluate(run_dir,spec,recipe,valid,held,out,name,scope):
    cp=run_dir/'selected.pt'; freeze_path=run_dir/'freeze.json'
    frozen=cv_require_frozen(freeze_path,spec,cp,valid)
    model=load_model(recipe,cp); predictions=predict_model(model,recipe,held); validate_predictions(predictions,held,'segment')
    ops=operating_metrics(predictions,held,{int(k):v for k,v in frozen['thresholds'].items()}); ops.to_csv(out/(name+'_operating.csv'),index=False)
    prediction_path=out/(name+'_predictions.jsonl.gz'); cv_save_predictions(predictions,prediction_path,'segment')
    record=dict(status='evaluated',name=name,spec=spec,model=recipe['name'],fold=spec['job']['fold'],seed=spec['job']['seed'],arm=spec['job']['arm'],freeze_sha=file_sha(freeze_path),checkpoint_sha=file_sha(cp),threshold_source='inner-valid',thresholds=frozen['thresholds'],evaluation_stems=sorted(held),box=canonical_coco(predictions,held,'bbox'),mask=canonical_coco(predictions,held,'segm'),predictions=str(prediction_path),predictions_sha=file_sha(prediction_path),scope=scope)
    cv_atomic_json(out/(name+'.json'),record)
    del predictions,model; gc.collect(); runtime()[0].cuda.empty_cache()
    return record


def run_full_cv(plan,token):
    assert not any([DO_TRAIN,DO_CV_SMOKE,DO_EVAL_TEST,DO_ABLATIONS,DO_EXPORT])
    assert sum([DO_FULL_CV,DO_CV_FREEZE,DO_CV_EVALUATE])==1, 'Train, freeze and evaluation are separate invocations'
    assert CV_APPROVED_PROTOCOL==token, 'Owner must approve exact protocol ID'
    previews=(EXP/'results/nb16_v2').glob('*/cv_preview_'+token+'.json')
    assert any(file_sha(p)==CV_APPROVED_PREVIEW_SHA and json.loads(p.read_text())['plan']==plan for p in previews), 'Approve production preview SHA before running'
    smokes=list((EXP/'results/nb16_v2').glob('*/cv_pipeline_smoke_'+token+'.json'))
    assert any(json.loads(p.read_text()).get('status')=='PASS' and json.loads(p.read_text()).get('plan')==plan for p in smokes), 'Run bounded production-batch smoke before full runs'
    torch,device=seed_everything(42); assert torch.cuda.is_available(); ensure_coco()
    out=EXP/'results/nb16_recipe_v2'/token; runs=EXP/'runs/nb16_recipe_v2'/token
    out.mkdir(parents=True,exist_ok=True); runs.mkdir(parents=True,exist_ok=True); cv_atomic_json(out/'plan.json',plan)
    assignments=pd.read_csv(CV_ROOT/'assignments.csv',keep_default_na=False); records=[]; errors=[]
    for job in plan['jobs']:
        recipe,fold,seed,arm=job['recipe'],job['fold'],job['seed'],job['arm']
        name=f'{recipe["name"]}_fold{fold}_seed{seed}_{arm}'; run_dir=runs/name; root=CV_ROOT/f'fold{fold}'/arm
        parents=assignments[(assignments.fold==fold)&(assignments.role=='train')]; steps=math.ceil(len(parents)/recipe['batch'])
        spec=dict(plan_id=token,job=job,steps_per_epoch=steps,optimizer_steps=steps*recipe['epochs'],initializer_sha=file_sha(EXP/'weights'/recipe['init']),cv_id=CV_HANDOFF['cv_id'],dataset_id=DATA_ID,source_id=plan['source_id'],train_parent_stems=sorted(parents.stem))
        try:
            run_dir.mkdir(parents=True,exist_ok=True); spec_path=run_dir/'spec.json'
            if spec_path.exists(): assert json.loads(spec_path.read_text())==spec
            else:
                assert not any(run_dir.iterdir()), 'Incomplete run without identity'
                cv_atomic_json(spec_path,spec)
            training_path=run_dir/'training.json'; valid=load_gt(root/'valid')
            assert set(valid)==set(assignments.loc[(assignments.fold==fold)&(assignments.role=='valid'),'stem'])
            if DO_FULL_CV:
                if training_path.is_file():
                    record=json.loads(training_path.read_text()); assert record['spec']==spec and record['checkpoint_sha']==file_sha(run_dir/'selected.pt')
                else:
                    cp,timing,state,init_sha=cv_train_yolo(recipe,seed,run_dir,root,valid,steps,spec)
                    curve=pd.read_csv(run_dir/'selection.csv'); assert curve.optimizer_steps.tolist()==[steps*(e+1) for e in range(recipe['epochs'])]
                    usage=CVParentSampler(cv_training_rows(fold,arm),seed,arm=='offline_aug'); usage.load_state_dict(state)
                    record=dict(status='trained; not evaluated',name=name,spec=spec,checkpoint=str(cp),checkpoint_sha=file_sha(cp),optimizer_steps=spec['optimizer_steps'],selected_epoch=int(curve.loc[curve.val_AP.idxmax(),'epoch']),training_init_sha=init_sha,timing=timing,sampling=usage.summary(),sampling_seed=seed,gpu=torch.cuda.get_device_name())
                    usage.table().to_csv(run_dir/'sampling_counts.csv',index=False); cv_atomic_json(training_path,record)
                counterpart=runs/f'{recipe["name"]}_fold{fold}_seed{seed}_{"control" if arm=="offline_aug" else "offline_aug"}'/'training.json'
                if counterpart.is_file():
                    other=json.loads(counterpart.read_text()); assert other['training_init_sha']==record['training_init_sha'] and other['optimizer_steps']==record['optimizer_steps'] and other['sampling']['output_images']==record['sampling']['output_images']
                records.append(record)
            elif DO_CV_FREEZE: records.append(cv_freeze(run_dir,spec,recipe,valid))
            else:
                cv_require_frozen(run_dir/'freeze.json',spec,run_dir/'selected.pt',valid)
                held=load_gt(root/'test'); assert set(held)==set(assignments.loc[(assignments.fold==fold)&(assignments.role=='heldout'),'stem']) and set(held).isdisjoint(valid)
                records.append(cv_evaluate(run_dir,spec,recipe,valid,held,out,name,plan['scope']))
        except Exception as error:
            import traceback
            failure=dict(name=name,error=type(error).__name__,message=str(error),traceback=traceback.format_exc()); errors.append(failure)
            cv_atomic_json(out/(name+'_error.json'),failure); print('FAILED',name,failure['message'])
        finally:
            gc.collect(); torch.cuda.empty_cache()
            cv_atomic_json(out/'progress.json',dict(stage='train' if DO_FULL_CV else 'freeze' if DO_CV_FREEZE else 'evaluate',completed=len(records),failed=len(errors),total=len(plan['jobs']),current=name,status='running'))
    cv_atomic_json(out/'progress.json',dict(completed=len(records),failed=len(errors),total=len(plan['jobs']),status='completed' if not errors and len(records)==len(plan['jobs']) else 'finished_with_errors'))
    if DO_CV_EVALUATE and records:
        table=pd.DataFrame([dict(model=r['model'],fold=r['fold'],seed=r['seed'],arm=r['arm'],mask_AP=r['mask']['AP'],box_AP=r['box']['AP']) for r in records]); table.to_csv(out/'results.csv',index=False)
        table.groupby(['model','arm'])[['mask_AP','box_AP']].agg(['mean','std','count']).to_csv(out/'fold_summary.csv')
    if errors: raise RuntimeError(f'{len(errors)} jobs failed; inspect progress/errors, do not report a complete comparison')
    return out

## 9. Preflight, external ZIP audit และ Stripe benchmark

In [11]:
DO_EXTERNAL_AUDIT = os.environ.get('NB16_EXTERNAL_AUDIT','0')=='1'
DO_EXTERNAL_EVAL = os.environ.get('NB16_EVAL_EXTERNAL','0')=='1'
DO_STRIPE_EVAL = os.environ.get('NB16_EVAL_STRIPE','0')=='1'
EXTERNAL_ZIPS = {'external_1':EXP/'data/Piled Materials.v2i.yolo26.zip','external_2':EXP/'data/S02.zip','external_3':EXP/'data/Separate Rock-Sand.v6i.yolo26.zip'}
EXTERNAL_REVIEWS = {}
EXTERNAL_RUN_DIR = None
STRIPE_TEST_ROOT = None


def cv_sampler_checks():
    rows=pd.DataFrame([dict(parent_stem=f'parent{i}',variant=v,image=f'{i}_{v}.png',role='train') for i in range(3) for v in CV_VARIANTS])
    aug=CVParentSampler(rows,42,True); base=CVParentSampler(rows[rows.variant=='original'],42,False)
    for _ in range(100):
        aug.draw(True); base.draw(True); assert aug.draws[-1]['parent']==base.draws[-1]['parent']
        aug.draw(False)
    state=aug.state_dict(); replay=CVParentSampler(rows,42,True); replay.load_state_dict(state)
    assert [aug.draw(True) for _ in range(20)]==[replay.draw(True) for _ in range(20)]
    assert aug.summary()==replay.summary()
    try: CVParentSampler(rows.iloc[:-1],42,True)
    except AssertionError: pass
    else: raise AssertionError('Incomplete variant lineage accepted')
    print('PASS parent-uniform pairing, donor separation, sampler resume and incomplete-lineage rejection')


def cv_pipeline_smoke(plan,token):
    assert not any([DO_FULL_CV,DO_CV_FREEZE,DO_CV_EVALUATE,DO_TRAIN,DO_INFER,DO_EVAL_TEST,DO_EXPORT,DO_EXTERNAL_EVAL,DO_STRIPE_EVAL])
    torch,_=runtime(); assert torch.cuda.is_available(); ensure_coco()
    root=EXP/'runs/nb16_recipe_v2_smoke'/STAMP; root.mkdir(parents=True,exist_ok=False)
    valid_all=load_gt(CV_ROOT/'fold0/control/valid'); chosen=smoke_parents(CV_LINEAGE[(CV_LINEAGE.fold==0)&(CV_LINEAGE.arm=='control')],'valid')
    valid={k:valid_all[k] for k in chosen}; records=[]
    for model_recipe in CV_FULL_RECIPES:
        pair=[]
        for arm in CV_ARMS:
            recipe=dict(model_recipe,epochs=1); job=dict(recipe=recipe,fold=0,seed=42,arm=arm)
            spec=dict(plan_id=token,job=job,steps_per_epoch=2,optimizer_steps=2,initializer_sha=file_sha(EXP/'weights'/recipe['init']),cv_id=CV_HANDOFF['cv_id'],dataset_id=DATA_ID,source_id=plan['source_id'],scope='bounded production-batch smoke; no outer predictions')
            run_dir=root/(recipe['name']+'_'+arm); run_dir.mkdir(); cv_atomic_json(run_dir/'spec.json',spec)
            torch.cuda.reset_peak_memory_stats()
            cp,timing,state,init_sha=cv_train_yolo(recipe,42,run_dir,CV_ROOT/'fold0'/arm,valid,2,spec)
            committed=cv_resume_load(run_dir/'resume.pt',spec)
            assert committed['cv_steps']==2 and committed['training_final_sha']!=init_sha
            sampler=CVParentSampler(cv_training_rows(0,arm),42,arm=='offline_aug'); sampler.load_state_dict(state)
            assert sampler.outputs==2*recipe['batch']
            record=dict(model=recipe['name'],arm=arm,optimizer_steps=2,output_images=sampler.outputs,sources=sampler.summary(),finite_nonzero_gradients=True,training_init_sha=init_sha,peak_allocated_gib=torch.cuda.max_memory_allocated()/2**30,checkpoint_sha=file_sha(cp),resume_sha=file_sha(run_dir/'resume.pt'),timing=timing,outer_predictions=0)
            pair.append(record); records.append(record); del committed; gc.collect(); torch.cuda.empty_cache()
        assert pair[0]['training_init_sha']==pair[1]['training_init_sha'] and pair[0]['output_images']==pair[1]['output_images']
    path=OUT/('cv_pipeline_smoke_'+token+'.json')
    cv_atomic_json(path,dict(status='PASS',plan=plan,records=records,outer_predictions=0,scope='preflight only; not recipe benefit'))
    print('PASS bounded production-batch smoke:',path)
    return path


def eval_image_record(name,raw,label):
    from shapely.geometry import Polygon
    image=cv2.imdecode(np.frombuffer(raw,np.uint8),cv2.IMREAD_COLOR)
    assert image is not None and image.size<=180000000, 'Invalid/oversized image'
    counts=[0,0]; h,w=image.shape[:2]
    for line in label.splitlines():
        if not line.strip(): continue
        fields=line.split(); assert len(fields)>=7 and (len(fields)-1)%2==0
        cls=int(fields[0]); xy=np.asarray(fields[1:],float).reshape(-1,2)
        assert cls in (0,1) and np.isfinite(xy).all() and ((xy>=0)&(xy<=1)).all()
        polygon=Polygon(xy); assert polygon.is_valid and polygon.area>0
        mask=np.zeros((h,w),np.uint8); cv2.fillPoly(mask,[(xy*[w,h]).astype(np.int32)],1); assert mask.any()
        counts[cls]+=1
    gray=cv2.resize(cv2.cvtColor(image,cv2.COLOR_BGR2GRAY),(9,8),interpolation=cv2.INTER_AREA)
    return dict(image=name,image_sha=hashlib.sha256(raw).hexdigest(),label_sha=hashlib.sha256(label.encode()).hexdigest(),decoded_sha=hashlib.sha256(image.tobytes()+str(image.shape).encode()).hexdigest(),dhash=int.from_bytes(np.packbits((gray[:,1:]>gray[:,:-1]).ravel()).tobytes(),'big'),gravel=counts[0],sand=counts[1],height=h,width=w)


def external_reference_records():
    records=[]
    for row in MANIFEST.itertuples(index=False):
        label=Path(row.path).parent.parent/'labels'/(row.stem+'.txt')
        record=eval_image_record(row.image,Path(row.path).read_bytes(),label.read_text())
        record.update(dataset='source_pool',split=row.split); records.append(record)
    return records


def audit_external_zip(name,path,references):
    import zipfile,yaml
    from pathlib import PurePosixPath
    path=Path(path); rows=[]; failures=[]; matches=[]
    with zipfile.ZipFile(path) as archive:
        infos=archive.infolist(); assert len({i.filename for i in infos})==len(infos), 'Duplicate ZIP members'
        assert sum(i.file_size for i in infos)<=2*2**30, 'ZIP too large for bounded audit'
        for info in infos:
            member=PurePosixPath(info.filename)
            assert not member.is_absolute() and '..' not in member.parts and '\\' not in info.filename
            assert info.file_size<=64*2**20 and (info.external_attr>>16)&0o170000!=0o120000
        members={i.filename for i in infos if not i.is_dir()}
        configs=[m for m in members if PurePosixPath(m).name=='data.yaml']; assert len(configs)==1, 'Ambiguous/missing ontology config'
        metadata=yaml.safe_load(archive.read(configs[0])); names=metadata['names']
        if isinstance(names,dict):
            assert sorted(map(int,names))==[0,1]
            names=[names[k] for k in sorted(names,key=int)]
        ontology_matches=names==CLS
        images=sorted(m for m in members if PurePosixPath(m).suffix.lower() in {'.jpg','.jpeg','.png','.bmp'} and PurePosixPath(m).parent.name=='images')
        expected_labels={str(PurePosixPath(m).parent.parent/'labels'/(PurePosixPath(m).stem+'.txt')) for m in images}
        actual_labels={m for m in members if PurePosixPath(m).parent.name=='labels' and m.endswith('.txt')}
        assert images and actual_labels==expected_labels, 'ZIP image/label mismatch'
        for image_name in images:
            label_name=str(PurePosixPath(image_name).parent.parent/'labels'/(PurePosixPath(image_name).stem+'.txt'))
            try:
                row=eval_image_record(image_name,archive.read(image_name),archive.read(label_name).decode('utf-8'))
                row['label_member']=label_name; rows.append(row)
            except (AssertionError,ValueError,UnicodeError) as error:
                failures.append(dict(image=image_name,error=str(error)))
    for row in rows:
        for ref in references+rows:
            if ref is row: continue
            distance=(row['dhash']^ref['dhash']).bit_count()
            exact=row['decoded_sha']==ref['decoded_sha'] or row['image_sha']==ref['image_sha']
            if exact or distance<=6: matches.append(dict(image=row['image'],reference=ref['image'],reference_dataset=ref.get('dataset',name),exact=exact,hamming=distance))
    identity=dict(name=name,zip_sha=file_sha(path),source_dataset_id=DATA_ID,source_id=CODE_ID,names=names,images=rows)
    return dict(audit_id=cv_json_id(identity),identity=identity,mechanical_pass=ontology_matches and not failures and len(rows)==len(images),ontology_matches=ontology_matches,annotation_failures=failures,duplicate_candidates=matches,ontology_review='PENDING',annotation_semantics_review='PENDING',duplicate_review='PENDING',independent_holdout=False)


def require_external_review(audit,review):
    assert audit['mechanical_pass'], 'External mechanical audit failed'
    assert review.get('audit_id')==audit['audit_id']
    assert all(review.get(k) is True for k in ['ontology_approved','annotations_approved','duplicate_review_approved'])
    assert review.get('reviewer') and review.get('evidence'), 'Record reviewer and adjudication evidence'
    return 'external audited benchmark; independent acquisition not established'


def external_run_context(run_dir):
    run_dir=Path(run_dir).resolve()
    assert run_dir.is_relative_to((EXP/'runs/nb16_recipe_v2').resolve())
    spec=json.loads((run_dir/'spec.json').read_text())
    assert spec['source_id'] in ACCEPTED_SOURCE_IDS and spec['dataset_id']==DATA_ID and spec['cv_id']==CV_HANDOFF['cv_id'] and spec['plan_id'] in ACCEPTED_PLAN_IDS|{CV_PROTOCOL_ID}
    recipe=spec['job']['recipe']; root=CV_ROOT/f'fold{spec["job"]["fold"]}'/spec['job']['arm']; valid=load_gt(root/'valid')
    cv_require_frozen(run_dir/'freeze.json',spec,run_dir/'selected.pt',valid)
    return run_dir,spec,recipe,valid


def evaluate_external_zip(name,path,audit,review,run_dir):
    import zipfile,tempfile
    scope=require_external_review(audit,review)
    assert file_sha(path)==audit['identity']['zip_sha']
    run_dir,spec,recipe,valid=external_run_context(run_dir)
    out=EXP/'results/nb16_recipe_v2'/spec['plan_id']/'external'/run_dir.name/name/audit['audit_id'][:20]
    out.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='aggregate-external-eval-') as temporary:
        root=Path(temporary); (root/'images').mkdir(); (root/'labels').mkdir()
        with zipfile.ZipFile(path) as archive:
            for row in audit['identity']['images']:
                stem=hashlib.sha256(row['image'].encode()).hexdigest()[:24]
                target=root/'images'/(stem+Path(row['image']).suffix.lower()); label=root/'labels'/(stem+'.txt')
                target.write_bytes(archive.read(row['image'])); label.write_bytes(archive.read(row['label_member']))
                assert file_sha(target)==row['image_sha'] and hashlib.sha256(label.read_text().encode()).hexdigest()==row['label_sha']
        record=cv_evaluate(run_dir,spec,recipe,valid,load_gt(root),out,name,scope)
    record.update(external_audit=audit,review=review,independent_holdout=False)
    cv_atomic_json(out/(name+'.json'),record)
    return record


def evaluate_stripe(run_dir,test_root):
    run_dir,spec,recipe,valid=external_run_context(run_dir); test_root=Path(test_root).resolve()
    images=sorted(p for p in (test_root/'images').iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'})
    assert len(images)==88 and len({p.stem for p in images})==88, 'Stripe must be the explicitly supplied 88-image benchmark'
    assert {p.stem for p in images}=={p.stem for p in (test_root/'labels').glob('*.txt')}
    assert not any('__cvaug_' in p.stem or p.stem.endswith(('_aug1','_aug2','_aug3')) for p in images)
    rows=[eval_image_record(p.name,p.read_bytes(),(test_root/'labels'/(p.stem+'.txt')).read_text()) for p in images]
    dataset_id=cv_json_id(rows); out=EXP/'results/nb16_recipe_v2'/spec['plan_id']/'stripe'/run_dir.name/dataset_id[:20]; out.mkdir(parents=True,exist_ok=True)
    record=cv_evaluate(run_dir,spec,recipe,valid,load_gt(test_root),out,'stripe_88','development-inspected, seen-scene test; not untouched')
    record.update(dataset_id=dataset_id,source_root=str(test_root),test_decision_history_audit='PENDING; no claim that scores were never used for development')
    cv_atomic_json(out/'stripe_88.json',record)
    return record


cv_sampler_checks()
if CV_HANDOFF is None:
    assert not any([DO_FULL_CV,DO_CV_PREVIEW,DO_CV_SMOKE,DO_CV_FREEZE,DO_CV_EVALUATE,DO_EXTERNAL_EVAL,DO_STRIPE_EVAL]), 'Missing v2 handoff; do not use v1'
    print('V2 pending 00 audit/build. No dataset creation/training/evaluation executed.')
else:
    if os.environ.get('NB16_ADOPT_PLAN','0')=='1':
        CV_PLAN=cv_adopt_plan(CV_APPROVED_PROTOCOL); CV_PROTOCOL_ID=CV_APPROVED_PROTOCOL
    else:
        CV_PLAN,CV_PROTOCOL_ID=cv_plan()
    ACCEPTED_PLAN_IDS.add(CV_PROTOCOL_ID)
    print('DRAFT protocol ID:',CV_PROTOCOL_ID,'| jobs:',len(CV_PLAN['jobs']))
    assert sum([DO_FULL_CV,DO_CV_PREVIEW,DO_CV_SMOKE,DO_CV_FREEZE,DO_CV_EVALUATE,DO_EXTERNAL_EVAL,DO_STRIPE_EVAL])<=1, 'One stage per invocation'
    if DO_CV_PREVIEW: CV_PREVIEW_PATH=cv_preview(CV_PLAN,CV_PROTOCOL_ID)
    if DO_CV_SMOKE: CV_SMOKE_PATH=cv_pipeline_smoke(CV_PLAN,CV_PROTOCOL_ID)
    if any([DO_FULL_CV,DO_CV_FREEZE,DO_CV_EVALUATE]): FULL_CV_OUT=run_full_cv(CV_PLAN,CV_PROTOCOL_ID)
if DO_EXTERNAL_AUDIT or DO_EXTERNAL_EVAL:
    references=external_reference_records(); audits={}
    for name,path in EXTERNAL_ZIPS.items():
        audit=audit_external_zip(name,path,references); audits[name]=audit; cv_atomic_json(OUT/(name+'_audit.json'),audit)
        references.extend(dict(row,dataset=name) for row in audit['identity']['images'])
    for name,audit in audits.items():
        if DO_EXTERNAL_EVAL:
            assert EXTERNAL_RUN_DIR is not None
            evaluate_external_zip(name,EXTERNAL_ZIPS[name],audit,EXTERNAL_REVIEWS.get(name,{}),EXTERNAL_RUN_DIR)
if DO_STRIPE_EVAL:
    assert EXTERNAL_RUN_DIR is not None and STRIPE_TEST_ROOT is not None
    evaluate_stripe(EXTERNAL_RUN_DIR,STRIPE_TEST_ROOT)

PASS parent-uniform pairing, donor separation, sampler resume and incomplete-lineage rejection
DRAFT protocol ID: 3b867e9545011a5f3c52 | jobs: 20


## 10. Batch probe (opt-in)

In [12]:
DO_BATCH_PROBE = os.environ.get('NB16_CV_PROBE','0') == '1'
CV_PROBE_STEPS = 3


def run_batch_probe():
    import shutil
    assert CV_HANDOFF is not None and len(CV_PREFLIGHT)==30
    assert not DO_FULL_CV and not DO_TRAIN and not DO_CV_SMOKE and not DO_EVAL_TEST and not DO_ABLATIONS, 'Use batch-probe opt-in alone'
    ensure_coco(); torch,device = seed_everything(42)
    assert torch.cuda.is_available(), 'Batch probe requires GPU'
    assignments = pd.read_csv(CV_ROOT/'assignments.csv',keep_default_na=False)
    root = CV_ROOT/'fold0'/'control'
    valid = load_gt(root/'valid')
    expected = set(assignments.loc[(assignments.fold==0)&(assignments.role=='valid'),'stem'])
    assert set(valid)==expected
    out = EXP/'results/nb16_batch_probe'/STAMP
    runs = EXP/'runs/nb16_batch_probe'/STAMP
    out.mkdir(parents=True,exist_ok=False); runs.mkdir(parents=True,exist_ok=False)
    rows=[]
    for recipe in CV_FULL_RECIPES:
        probe_recipe = dict(recipe,epochs=1)
        run_dir = runs/recipe['name']; run_dir.mkdir(parents=True,exist_ok=False)
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        try:
            if recipe['kind']=='yolo': cv_train_yolo(probe_recipe,42,run_dir,root,valid,CV_PROBE_STEPS)
            else: cv_train_torch(probe_recipe,42,run_dir,root,valid,CV_PROBE_STEPS)
            status='PASS'; error=None
        except Exception as exc:
            status='FAIL'; error=f'{type(exc).__name__}: {exc}'
            print('BATCH PROBE FAILED',recipe['name'],error,flush=True)
        rows.append(dict(model=recipe['name'],batch=recipe['batch'],status=status,peak_allocated_gib=float(torch.cuda.max_memory_allocated()/2**30),peak_reserved_gib=float(torch.cuda.max_memory_reserved()/2**30),seconds=time.perf_counter()-started,error=error))
        write_json(out/(recipe['name']+'.json'),rows[-1])
        gc.collect(); torch.cuda.empty_cache()
    table = pd.DataFrame(rows)
    table.to_csv(out/'batch_probe.csv',index=False)
    write_json(out/'batch_probe.json',dict(recipes=CV_FULL_RECIPES,steps=CV_PROBE_STEPS,results=rows))
    display(table)
    if table.status.ne('PASS').any(): raise RuntimeError('At least one batch probe failed; lower the failed recipe batch before full CV.')
    return table


if DO_BATCH_PROBE:
    BATCH_PROBE_TABLE = run_batch_probe()
else:
    print('Batch probe disabled. Set NB16_CV_PROBE=1 to validate per-recipe batches in notebook before full CV.')

Batch probe disabled. Set NB16_CV_PROBE=1 to validate per-recipe batches in notebook before full CV.
